# Clasificación de 3 escenas con machine learning clásico

Este cuaderno reconstruye la línea base de **machine learning** "de inicio de
curso" sobre el mismo dataset y el mismo split que los modelos de aprendizaje
profundo, para poder comparar sobre la base del enunciado. No se usan píxeles
en crudo, sino dos descriptores concatenados:

* **Histograma de color HSV** con cubos `[8, 8, 8]` (tono, saturación, valor),
  normalizado a suma 1 → 512 características. La costa es dominante en azules,
  el bosque en verdes y la autopista en grises: el color separa bien las tres
  escenas.
* **HOG** sobre la imagen en escala de grises (`orientations=9`,
  `pixels_per_cell=(16, 16)`, `cells_per_block=(2, 2)`) → 1764 características
  de gradiente y textura (líneas de la calzada, horizonte, troncos y copas).

En total, 2276 características por imagen. Tres clasificadores, cada uno en su
propia carpeta:

| Modelo | Carpeta | Notas |
|---|---|---|
| Regresión logística | `resultados/ml/logreg/` | `Pipeline` con `StandardScaler` + `LogisticRegression(max_iter=2000)`. |
| SVM (RBF) | `resultados/ml/svm/` | `SVC(kernel="rbf", C=10, probability=True)`. |
| Random Forest | `resultados/ml/rf/` | `RandomForestClassifier(n_estimators=300, random_state=SEED)`. |

Los tres modelos se ajustan **solo con el train** del split común (los modelos
de sklearn no usan early stopping, así que la partición de validación no
interviene) y se evalúan sobre el mismo `test_idx` que los modelos de deep
learning. Los modelos entrenados se guardan como `model.joblib` y se cargan por
defecto; cambia `FORCE_RETRAIN = True` para reentrenarlos.

## Configuración

In [ ]:
import time

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

import common_3scenes as cs

FORCE_RETRAIN = False
sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(cs.SEED)

## Carga del dataset y split común

In [ ]:
data = cs.load_split_data()
y_train = data["y_train"]
y_test = data["y_test"]
test_paths = data["test_paths"]

print("Imágenes -> train:", data["X_train"].shape, "| val:", data["X_val"].shape, "| test:", data["X_test"].shape)
print("Distribución de test:", {name: int(np.sum(y_test == name)) for name in cs.CLASSES})

## Descriptores: histograma HSV + HOG

Para cada imagen RGB de 128×128 se concatena:

1. El histograma HSV con 8 cubos por canal (`cv2.calcHist` con `[8, 8, 8]`),
   aplanado y normalizado a suma 1.
2. El descriptor HOG de la versión en gris (`skimage.feature.hog` con 9
   orientaciones, celdas de 16×16 píxeles y bloques de 2×2 celdas), ya
   aplanado.

La partición de validación no se describe porque los modelos clásicos no la
usan (no hay early stopping); el test se describe solo para evaluar.

In [ ]:
def describe_image(image):
    """Return the concatenated HSV histogram + HOG descriptor of an RGB image."""
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    histogram = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
    histogram = histogram.flatten()
    histogram = histogram / (histogram.sum() + 1e-8)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
    )
    return np.concatenate([histogram, hog_features]).astype(np.float32)


def describe_images(images):
    """Describe a batch of RGB images."""
    return np.asarray([describe_image(image) for image in images])


X_train_features = describe_images(data["X_train"])
X_test_features = describe_images(data["X_test"])

print("Descriptores -> train:", X_train_features.shape, "| test:", X_test_features.shape)

## Modelos: ajuste, evaluación y caché

Los tres clasificadores se entrenan con `cs.load_or_train`, igual que los
modelos de deep learning: si `resultados/ml/<modelo>/` ya tiene `metrics.json`
y `model.joblib`, se cargan; si no, se entrenan y se guardan
`metrics.json`, `predictions.csv`, `classification_report.txt` y
`model.joblib` (no hay `history.csv`: los modelos clásicos no tienen épocas).

Si un clasificador no pudiera producir probabilidades, `roc_auc_ovr_macro`
quedaría a `None` y se indicaría en `notes`; los tres elegidos sí tienen
`predict_proba`.

In [ ]:
DESCRIPTOR = "HSV histogram [8,8,8] + HOG(9, (16,16), (2,2))"


def build_logreg():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2000)),
        ]
    )


def build_svm():
    return SVC(kernel="rbf", C=10, probability=True)


def build_rf():
    return RandomForestClassifier(n_estimators=300, random_state=cs.SEED)


def fit_sklearn(model):
    started = time.perf_counter()
    model.fit(X_train_features, y_train)
    seconds = time.perf_counter() - started
    return model, None, seconds


def predict_sklearn(model, X):
    """Predict labels and probabilities ordered like ``cs.CLASSES``."""
    predicted = model.predict(X)
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        columns = [list(model.classes_).index(name) for name in cs.CLASSES]
        probabilities = probabilities[:, columns]
    else:
        probabilities = None
    return predicted, probabilities


def make_evaluate(folder, name, hyperparameters, notes):
    def evaluate(model, history, training_seconds):
        started = time.perf_counter()
        predicted, probabilities = predict_sklearn(model, X_test_features)
        inference_ms = 1000 * (time.perf_counter() - started) / len(X_test_features)
        if probabilities is None:
            notes_with_probability = notes + " Without predict_proba: ROC-AUC not computed."
        else:
            notes_with_probability = notes
        metrics = cs.compute_metrics(
            name=name,
            family="machine learning",
            y_true=y_test,
            y_pred=predicted,
            y_prob=probabilities,
            training_seconds=training_seconds,
            inference_ms=inference_ms,
            hyperparameters=hyperparameters,
            notes=notes_with_probability,
        )
        cs.save_results(
            folder,
            metrics,
            y_test,
            predicted,
            probabilities,
            test_paths=test_paths,
            history=None,
            model=None,
        )
        joblib.dump(model, cs.result_dir(folder) / cs.ARTIFACTS["sklearn_model"])
        return metrics

    return evaluate


def load_sklearn(folder):
    return joblib.load(cs.result_dir(folder) / cs.ARTIFACTS["sklearn_model"])


evaluate_logreg = make_evaluate(
    "ml/logreg",
    "Regresión logística",
    {
        "descriptor": DESCRIPTOR,
        "classifier": "LogisticRegression",
        "max_iter": 2000,
        "scaling": "StandardScaler",
    },
    "Linear baseline on HSV+HOG descriptors inside a scaled pipeline.",
)
evaluate_svm = make_evaluate(
    "ml/svm",
    "SVM (RBF)",
    {
        "descriptor": DESCRIPTOR,
        "classifier": "SVC",
        "kernel": "rbf",
        "C": 10,
        "probability": True,
    },
    "RBF-kernel SVM on HSV+HOG descriptors with Platt-calibrated probabilities.",
)
evaluate_rf = make_evaluate(
    "ml/rf",
    "Random Forest",
    {
        "descriptor": DESCRIPTOR,
        "classifier": "RandomForestClassifier",
        "n_estimators": 300,
    },
    "Random forest of 300 trees on HSV+HOG descriptors.",
)

In [ ]:
logreg_model, _, metrics_logreg, retrained_logreg = cs.load_or_train(
    "ml/logreg",
    build_model=build_logreg,
    fit_model=fit_sklearn,
    evaluate_model=evaluate_logreg,
    load_model=load_sklearn,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics_logreg["accuracy"], 4))

In [ ]:
svm_model, _, metrics_svm, retrained_svm = cs.load_or_train(
    "ml/svm",
    build_model=build_svm,
    fit_model=fit_sklearn,
    evaluate_model=evaluate_svm,
    load_model=load_sklearn,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics_svm["accuracy"], 4))

In [ ]:
rf_model, _, metrics_rf, retrained_rf = cs.load_or_train(
    "ml/rf",
    build_model=build_rf,
    fit_model=fit_sklearn,
    evaluate_model=evaluate_rf,
    load_model=load_sklearn,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics_rf["accuracy"], 4))

## Comparación de los tres modelos

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "model": metrics["name"],
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "cohen_kappa": metrics["cohen_kappa"],
            "roc_auc_ovr_macro": metrics["roc_auc_ovr_macro"],
            "training_seconds": metrics["training_seconds"],
            "inference_ms_per_image": metrics["inference_ms_per_image"],
        }
        for metrics in [metrics_logreg, metrics_svm, metrics_rf]
    ]
)
display = comparison.rename(
    columns={
        "model": "Modelo",
        "accuracy": "Accuracy",
        "f1_macro": "F1 macro",
        "cohen_kappa": "Kappa",
        "roc_auc_ovr_macro": "AUC OVR",
        "training_seconds": "Entrenamiento (s)",
        "inference_ms_per_image": "Inferencia (ms/img)",
    }
)
display.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
positions = np.arange(len(comparison))
width = 0.36
ax.bar(positions - width / 2, comparison["accuracy"], width, label="Accuracy")
ax.bar(positions + width / 2, comparison["f1_macro"], width, label="F1 macro")
ax.set_xticks(positions, comparison["model"])
ax.set_ylim(0, 1.05)
ax.set_title("Modelos de machine learning (HSV + HOG)")
ax.legend()
for index, (accuracy, f1_value) in enumerate(
    zip(comparison["accuracy"], comparison["f1_macro"])
):
    ax.text(index - width / 2, accuracy + 0.01, f"{accuracy:.3f}", ha="center", fontsize=9)
    ax.text(index + width / 2, f1_value + 0.01, f"{f1_value:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## Observaciones

* Color (histograma HSV) y textura/gradiente (HOG) son un descriptor razonable
  para estas tres escenas: la discriminación costa/bosque/autopista cae en gran
  parte en el color dominante y en las estructuras de bordes y líneas.
* La SVM con kernel RBF suele ser muy competitiva con unos 720 ejemplos de
  train y 2276 características; el bosque aleatorio es el más robusto al
  escalado y el más lento de inferir (300 árboles); la regresión logística es
  la línea base interpretable con regularización L2 implícita.
* Los números exactos de cada modelo quedan en la tabla de arriba y en
  `resultados/ml/<modelo>/metrics.json`; la comparación con las CNN y con
  DenseNet121 (IC 95 %, McNemar, matrices de confusión) está en `reto-1.ipynb`.
* `FORCE_RETRAIN = False` reutiliza los `model.joblib` ya guardados, así que
  revisar el cuaderno no reentrena.